# 09o — EGFR–MOL00484: repliche CUDA bound e persistenza alla native EGF–EGFR interface

Questa analisi estende la validazione bound di MOL00484 effettuata in 09l mediante repliche stocastiche più lunghe del complesso EGFR–MOL00484.

Le repliche partono dallo stesso stato finale della produzione ufficiale 09l da 1 ns. Utilizzano lo stesso sistema fisico, la stessa parametrizzazione del ligando e lo stesso microstato neutro; vengono assegnate nuove velocità e nuovi semi casuali. Le repliche costituiscono quindi rami stocastici della stessa preparazione strutturale e non preparazioni indipendenti del complesso.

Protocollo previsto:
- smoke test: 1 replica × 50 ps;
- produzione: 3 repliche × 10 ns;
- CUDA, precisione mixed;
- 300 K, 1 bar;
- timestep 2 fs;
- riequilibrazione NPT prima della produzione;
- nessuna nuova parametrizzazione.

Obiettivo principale site-specific:
verificare se MOL00484 mantiene nel tempo la localizzazione presso la native EGF–EGFR interface già osservata in 09l.

La valutazione site-specific utilizza esplicitamente i 43 residui della native EGF–EGFR interface adottati nell'audit A26b/09l. La semplice permanenza del ligando vicino a EGFR non viene considerata equivalente alla permanenza presso questa interfaccia.

09o caratterizza stabilità geometrica, persistenza dei contatti e comportamento dinamico della configurazione simulata. Non fornisce una stima rigorosa di ΔG, Kd, Ki o attività inibitoria e non costituisce validazione sperimentale.

In [12]:
from pathlib import Path
import json
import hashlib
import sys
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import openmm as mm
from openmm import app, unit

# ============================================================
# 09o — CONFIGURATION / PREFLIGHT
# No MD is started in this cell.
# ============================================================

PEARL_ROOT = Path(r"C:\Users\Roberto\PEARL")

SOURCE_ROOT = PEARL_ROOT / "MOL00484_09l_WINDOWS_1ns"

SOURCE_RUN = (
    SOURCE_ROOT
    / "production_runs"
    / "20261027T171745_698045Z_seed20261027"
)

FREE_RUN = (
    PEARL_ROOT
    / "outputs"
    / "pipeline_5_09n_mol00484_free_water"
    / "20261028T080940_432915Z_production"
)

OUTPUT_BASE = (
    PEARL_ROOT
    / "outputs"
    / "pipeline_5_09o_mol00484_bound_replicates"
)

# ------------------------------------------------------------
# Execution mode
# ------------------------------------------------------------

RUN_MD = True
FAST_TEST_MODE = False

SEEDS = [20261101] if FAST_TEST_MODE else [
    20261101,
    20261102,
    20261103,
]

EQUIL_PS = 10.0 if FAST_TEST_MODE else 100.0
PRODUCTION_PS = 50.0 if FAST_TEST_MODE else 10000.0
FRAME_PS = 1.0 if FAST_TEST_MODE else 10.0

PLATFORM = "CUDA"
PLATFORM_PROPERTIES = {
    "DeviceIndex": "0",
    "Precision": "mixed",
}

DISCARD_FRACTION = 0.20

# ------------------------------------------------------------
# Native EGF–EGFR interface used in A26b / 09l
# ------------------------------------------------------------

EGF_INTERFACE_RESIDUES_B = {
    "ALA415", "ALA68", "ARG29", "ARG353",
    "ASN12", "ASN128", "ASP22", "ASP355",
    "GLN16", "GLN384", "GLN408", "GLN411",
    "GLU90", "GLY18", "GLY441", "HIS346",
    "HIS409", "ILE438", "ILE467", "LEU14",
    "LEU17", "LEU325", "LEU348", "LEU382",
    "LEU69", "LEU98", "LYS13", "LYS465",
    "PHE357", "PHE412", "PRO349", "SER11",
    "SER356", "SER418", "SER440", "SER468",
    "SER99", "THR15", "THR358", "TYR101",
    "TYR45", "VAL350", "VAL417",
}

INTERFACE_CUTOFF_NM = 0.45

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256(path):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def dump(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, default=str),
        encoding="utf-8"
    )

print("=== 09o CONFIGURATION ===")
print("Candidate       : MOL00484")
print("09l source      :", SOURCE_RUN)
print("09n free source :", FREE_RUN)
print("Output base     :", OUTPUT_BASE)
print("FAST_TEST_MODE  :", FAST_TEST_MODE)
print("RUN_MD          :", RUN_MD)
print("Replicas        :", len(SEEDS))
print("Production/rep  :", PRODUCTION_PS, "ps")
print("Interface set   :", len(EGF_INTERFACE_RESIDUES_B), "residues")

=== 09o CONFIGURATION ===
Candidate       : MOL00484
09l source      : C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027
09n free source : C:\Users\Roberto\PEARL\outputs\pipeline_5_09n_mol00484_free_water\20261028T080940_432915Z_production
Output base     : C:\Users\Roberto\PEARL\outputs\pipeline_5_09o_mol00484_bound_replicates
FAST_TEST_MODE  : False
RUN_MD          : True
Replicas        : 3
Production/rep  : 10000.0 ps
Interface set   : 43 residues


In [6]:
# ============================================================
# 09o — CELL 2
# SOURCE FILE PREFLIGHT / PROVENANCE
# No MD is started in this cell.
# ============================================================

print("=== 09o SOURCE PREFLIGHT ===")

# ------------------------------------------------------------
# 09l authoritative bound source
# ------------------------------------------------------------

SOURCE_FILES = {
    "production_system":
        SOURCE_ROOT / "states" / "production_system.xml",

    "production_integrator":
        SOURCE_ROOT / "states" / "production_integrator.xml",

    "production_final":
        SOURCE_RUN / "states" / "production_final.xml",

    "solvated_pdb":
        SOURCE_ROOT / "structures" / "solvated.pdb",

    "solute_reference":
        SOURCE_ROOT / "structures" / "solute_reference_09f.pdb",

    "solute_atom_map":
        SOURCE_ROOT / "tables" / "solute_atom_map.csv",
}
# ------------------------------------------------------------
# 09n authoritative free-water results
# ------------------------------------------------------------

FREE_FILES = {
    "conformation_summary":
        FREE_RUN / "conformation_summary.csv",

    "hydration_summary":
        FREE_RUN / "hydration_summary.csv",

    "qc":
        FREE_RUN / "09n_QC.csv",

    "summary":
        FREE_RUN / "09n_summary.json",
}

# ------------------------------------------------------------
# Existence checks
# ------------------------------------------------------------

print("\n09l authoritative source:")
for name, path in SOURCE_FILES.items():
    ok = path.is_file()
    print(f"  {name:22s}: {'OK' if ok else 'MISSING'}")
    print(f"    {path}")
    require(ok, f"09l source file missing: {path}")

print("\n09n free-water source:")
for name, path in FREE_FILES.items():
    ok = path.is_file()
    print(f"  {name:22s}: {'OK' if ok else 'MISSING'}")
    print(f"    {path}")
    require(ok, f"09n source file missing: {path}")

# ------------------------------------------------------------
# SHA256 fingerprints
# ------------------------------------------------------------

print("\n09l SHA256:")
SOURCE_SHA256 = {}

for name, path in SOURCE_FILES.items():
    digest = sha256(path)
    SOURCE_SHA256[name] = digest
    print(f"  {name:22s}: {digest}")

print("\n09n SHA256:")
FREE_SHA256 = {}

for name, path in FREE_FILES.items():
    digest = sha256(path)
    FREE_SHA256[name] = digest
    print(f"  {name:22s}: {digest}")

# ------------------------------------------------------------
# Validate 09n summary
# ------------------------------------------------------------

free_summary = json.loads(
    FREE_FILES["summary"].read_text(encoding="utf-8")
)

require(
    free_summary.get("candidate") == "MOL00484",
    "09n candidate mismatch."
)

require(
    free_summary.get("free_water_replicas") == 3,
    "09n must contain 3 free-water replicas."
)

require(
    float(free_summary.get("production_ns_per_replica")) == 10.0,
    "09n replicas must be 10 ns each."
)

require(
    bool(free_summary.get("technical_qc_pass")),
    "09n technical QC did not pass."
)

require(
    free_summary.get("microstate") ==
    "neutral_same_as_authoritative_09l",
    "09n/09l microstate mismatch."
)

# ------------------------------------------------------------
# Basic 09l system checks
# ------------------------------------------------------------

system = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_system"].read_text(encoding="utf-8")
)

integrator = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_integrator"].read_text(encoding="utf-8")
)

final_state = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_final"].read_text(encoding="utf-8")
)

pdb = app.PDBFile(str(SOURCE_FILES["solvated_pdb"]))

print("\n09l system:")
print("  System particles :", system.getNumParticles())
print("  PDB atoms        :", pdb.topology.getNumAtoms())
print("  Integrator       :", type(integrator).__name__)

require(
    system.getNumParticles() == pdb.topology.getNumAtoms(),
    "09l System/PDB particle mismatch."
)

require(
    isinstance(integrator, mm.LangevinMiddleIntegrator),
    "Unexpected 09l integrator."
)

dt_ps = integrator.getStepSize().value_in_unit(unit.picosecond)

print("  timestep         :", dt_ps, "ps")

require(
    abs(dt_ps - 0.002) < 1e-10,
    "Unexpected timestep."
)

# Check barostat
barostats = [
    f for f in system.getForces()
    if isinstance(f, mm.MonteCarloBarostat)
]

print("  active barostats :", len(barostats))

require(
    len(barostats) == 1,
    "Expected exactly one MonteCarloBarostat."
)

require(
    barostats[0].getFrequency() > 0,
    "MonteCarloBarostat is not active."
)

# No residual positional restraint
external_forces = [
    f for f in system.getForces()
    if isinstance(f, mm.CustomExternalForce)
]

print("  CustomExternalForce:", len(external_forces))

require(
    len(external_forces) == 0,
    "Residual CustomExternalForce detected."
)

print("\n09n summary:")
print("  candidate       :", free_summary["candidate"])
print("  replicas        :", free_summary["free_water_replicas"])
print("  ns / replica    :", free_summary["production_ns_per_replica"])
print("  total free ns   :", free_summary["total_free_water_production_ns"])
print("  technical QC    :", free_summary["technical_qc_pass"])
print("  microstate      :", free_summary["microstate"])

print("\n09o SOURCE PREFLIGHT: OK")

=== 09o SOURCE PREFLIGHT ===

09l authoritative source:
  production_system     : OK
    C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\states\production_system.xml
  production_integrator : OK
    C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\states\production_integrator.xml
  production_final      : OK
    C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027\states\production_final.xml
  solvated_pdb          : OK
    C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\structures\solvated.pdb
  solute_reference      : OK
    C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\structures\solute_reference_09f.pdb
  solute_atom_map       : OK
    C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\tables\solute_atom_map.csv

09n free-water source:
  conformation_summary  : OK
    C:\Users\Roberto\PEARL\outputs\pipeline_5_09n_mol00484_free_water\20261028T080940_432915Z_production\conformation_summary.csv
  hydration_summary     : OK
    C:\Users\Rob

In [7]:
# ============================================================
# 09o — CELL 3
# SOLUTE / LIGAND / EGF–EGFR INTERFACE PREFLIGHT
# No MD is started in this cell.
# ============================================================

print("=== 09o STRUCTURAL / INTERFACE PREFLIGHT ===")

atom_map = pd.read_csv(SOURCE_FILES["solute_atom_map"])

print("\nAtom-map columns:")
print(list(atom_map.columns))

print("\nAtom-map:")
print("  rows       :", len(atom_map))
print("  index min  :", atom_map["index"].min())
print("  index max  :", atom_map["index"].max())

require(
    len(atom_map) == 9332,
    f"Unexpected solute atom count: {len(atom_map)}"
)

require(
    atom_map["index"].min() == 0 and atom_map["index"].max() == 9331,
    "Unexpected solute atom-index range."
)

print("\nComponents:")
print(atom_map["component"].value_counts())

require(
    (atom_map["component"] == "EGFR").sum() == 9286,
    "Unexpected EGFR atom count."
)

require(
    (atom_map["component"] == "MOL00484").sum() == 46,
    "Unexpected MOL00484 atom count."
)

# ------------------------------------------------------------
# Load only the 9332-atom solute reference
# ------------------------------------------------------------

import mdtraj as mt

ref = mt.load(str(SOURCE_FILES["solute_reference"]))

print("\nSolute reference:")
print("  atoms    :", ref.n_atoms)
print("  residues :", ref.n_residues)
print("  chains   :", ref.n_chains)

require(
    ref.n_atoms == len(atom_map),
    "Solute reference / atom-map mismatch."
)

# ------------------------------------------------------------
# Ligand selection
# Use authoritative component map rather than assuming resname.
# ------------------------------------------------------------

ligand_atoms = atom_map.loc[
    atom_map["component"] == "MOL00484",
    "index"
].to_numpy(dtype=int)

require(
    len(ligand_atoms) == 46,
    "MOL00484 must contain 46 atoms."
)

ligand_heavy = np.array([
    i for i in ligand_atoms
    if ref.topology.atom(int(i)).element is not None
    and ref.topology.atom(int(i)).element.symbol != "H"
], dtype=int)

print("\nMOL00484:")
print("  total atoms :", len(ligand_atoms))
print("  heavy atoms :", len(ligand_heavy))

require(
    len(ligand_heavy) == 25,
    f"Expected 25 MOL00484 heavy atoms, found {len(ligand_heavy)}."
)

# ------------------------------------------------------------
# EGFR atoms from authoritative component map
# ------------------------------------------------------------

egfr_atoms = atom_map.loc[
    atom_map["component"] == "EGFR",
    "index"
].to_numpy(dtype=int)

require(
    len(egfr_atoms) == 9286,
    "EGFR must contain 9286 atoms."
)

# ------------------------------------------------------------
# Identify the 43 native EGF–EGFR interface residues
# in the actual 09l solute topology.
#
# Keys are built as RESNAME + residue sequence number,
# e.g. HIS409, GLN408, SER11.
# ------------------------------------------------------------

interface_residue_atoms = {}
observed_egfr_residues = set()

for residue in ref.topology.residues:

    residue_atom_indices = np.array(
        [a.index for a in residue.atoms],
        dtype=int
    )

    # residue belongs to EGFR only if it contains authoritative
    # EGFR component atoms
    if not np.isin(residue_atom_indices, egfr_atoms).any():
        continue

    key = f"{residue.name}{residue.resSeq}"
    observed_egfr_residues.add(key)

    if key in EGF_INTERFACE_RESIDUES_B:
        heavy_indices = [
            a.index for a in residue.atoms
            if a.element is not None and a.element.symbol != "H"
        ]

        interface_residue_atoms[key] = np.array(
            heavy_indices,
            dtype=int
        )

found_interface = set(interface_residue_atoms)
missing_interface = EGF_INTERFACE_RESIDUES_B - found_interface
unexpected_interface = found_interface - EGF_INTERFACE_RESIDUES_B

print("\nNative EGF–EGFR interface:")
print("  expected residues :", len(EGF_INTERFACE_RESIDUES_B))
print("  found residues    :", len(found_interface))
print("  missing residues  :", sorted(missing_interface))
print("  unexpected        :", sorted(unexpected_interface))

require(
    found_interface == EGF_INTERFACE_RESIDUES_B,
    "Native EGF–EGFR interface residue mapping failed."
)

interface_heavy = np.array(
    sorted({
        int(i)
        for indices in interface_residue_atoms.values()
        for i in indices
    }),
    dtype=int
)

print("  interface heavy atoms :", len(interface_heavy))

require(
    len(interface_heavy) == 344,
    f"Expected 344 interface heavy atoms, found {len(interface_heavy)}."
)

# ------------------------------------------------------------
# Pair count for A26b-like analysis
# ------------------------------------------------------------

n_pairs = len(interface_heavy) * len(ligand_heavy)

print("\nA26b pair geometry:")
print("  interface heavy atoms :", len(interface_heavy))
print("  ligand heavy atoms    :", len(ligand_heavy))
print("  atom pairs            :", n_pairs)
print("  contact cutoff        :", INTERFACE_CUTOFF_NM, "nm")

require(
    n_pairs == 8600,
    f"Expected 8600 interface–ligand heavy-atom pairs, found {n_pairs}."
)

print("\n09o STRUCTURAL / INTERFACE PREFLIGHT: OK")

=== 09o STRUCTURAL / INTERFACE PREFLIGHT ===

Atom-map columns:
['index', 'name', 'element', 'residue_index', 'residue_name', 'residue_id', 'insertion_code', 'chain', 'component']

Atom-map:
  rows       : 9332
  index min  : 0
  index max  : 9331

Components:
component
EGFR        9286
MOL00484      46
Name: count, dtype: int64

Solute reference:
  atoms    : 9332
  residues : 615
  chains   : 2

MOL00484:
  total atoms : 46
  heavy atoms : 25

Native EGF–EGFR interface:
  expected residues : 43
  found residues    : 43
  missing residues  : []
  unexpected        : []
  interface heavy atoms : 344

A26b pair geometry:
  interface heavy atoms : 344
  ligand heavy atoms    : 25
  atom pairs            : 8600
  contact cutoff        : 0.45 nm

09o STRUCTURAL / INTERFACE PREFLIGHT: OK


In [8]:
# ============================================================
# 09o — CELL 4
# CUDA CONTEXT / 09l FINAL-STATE TRANSFER QC
# No production MD is run in this cell.
# ============================================================

print("=== 09o CUDA CONTEXT / STATE TRANSFER QC ===")

require(
    FAST_TEST_MODE is True,
    "This QC cell must first be run in FAST_TEST_MODE=True."
)

# Fresh copies: do not modify the objects used in preflight
test_system = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_system"].read_text(encoding="utf-8")
)

test_integrator = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_integrator"].read_text(encoding="utf-8")
)

test_state = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_final"].read_text(encoding="utf-8")
)

require(
    test_system.getNumParticles() == pdb.topology.getNumAtoms(),
    "System/topology mismatch."
)

require(
    isinstance(test_integrator, mm.LangevinMiddleIntegrator),
    "Unexpected integrator."
)

# Dedicated seed for technical CUDA QC
QC_SEED = 20261028

test_integrator.setRandomNumberSeed(QC_SEED)

test_barostats = [
    f for f in test_system.getForces()
    if isinstance(f, mm.MonteCarloBarostat)
]

require(
    len(test_barostats) == 1,
    "Expected exactly one barostat."
)

test_barostats[0].setRandomNumberSeed(QC_SEED + 100)

cuda = mm.Platform.getPlatformByName("CUDA")

print("Creating CUDA context...")

test_sim = app.Simulation(
    pdb.topology,
    test_system,
    test_integrator,
    cuda,
    PLATFORM_PROPERTIES
)

device = cuda.getPropertyValue(
    test_sim.context,
    "DeviceName"
)

print("CUDA device :", device)

# ------------------------------------------------------------
# IMPORTANT:
# Do NOT use context.setState(test_state).
#
# Transfer only:
#   - periodic box
#   - positions
#
# New velocities will be generated for each 09o stochastic branch.
# ------------------------------------------------------------

test_sim.context.setPeriodicBoxVectors(
    *test_state.getPeriodicBoxVectors()
)

test_sim.context.setPositions(
    test_state.getPositions()
)

# Get state BEFORE generating new velocities
state_before = test_sim.context.getState(
    getPositions=True,
    getEnergy=True
)

pe_before = state_before.getPotentialEnergy().value_in_unit(
    unit.kilojoule_per_mole
)

positions_before = state_before.getPositions(
    asNumpy=True
).value_in_unit(unit.nanometer)

require(
    np.isfinite(pe_before),
    "Potential energy is not finite."
)

require(
    np.isfinite(positions_before).all(),
    "Non-finite coordinates detected."
)

print("Potential energy after transfer :", pe_before, "kJ/mol")
print("Coordinates finite              :", True)

# ------------------------------------------------------------
# Generate NEW velocities for the stochastic branch
# ------------------------------------------------------------

test_sim.context.setVelocitiesToTemperature(
    test_integrator.getTemperature(),
    QC_SEED + 200
)

state_after = test_sim.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True
)

pe_after = state_after.getPotentialEnergy().value_in_unit(
    unit.kilojoule_per_mole
)

ke_after = state_after.getKineticEnergy().value_in_unit(
    unit.kilojoule_per_mole
)

velocities = state_after.getVelocities(
    asNumpy=True
).value_in_unit(unit.nanometer / unit.picosecond)

require(
    np.isfinite(pe_after),
    "Potential energy after velocity assignment is not finite."
)

require(
    np.isfinite(ke_after),
    "Kinetic energy is not finite."
)

require(
    np.isfinite(velocities).all(),
    "Non-finite velocities detected."
)

print("Potential energy               :", pe_after, "kJ/mol")
print("Kinetic energy                 :", ke_after, "kJ/mol")
print("Velocities finite              :", True)

print("\n09o CUDA CONTEXT / STATE TRANSFER QC: OK")

=== 09o CUDA CONTEXT / STATE TRANSFER QC ===
Creating CUDA context...
CUDA device : NVIDIA GeForce RTX 4090
Potential energy after transfer : -4305893.272434168 kJ/mol
Coordinates finite              : True
Potential energy               : -4305893.272492651 kJ/mol
Kinetic energy                 : 788924.2116669171 kJ/mol
Velocities finite              : True

09o CUDA CONTEXT / STATE TRANSFER QC: OK


In [9]:
# ============================================================
# 09o — CELL 5
# CUDA SMOKE TEST
# 10 ps NPT equilibration + 50 ps production
# One stochastic branch only
# ============================================================

import time
import gc

print("=== 09o CUDA SMOKE TEST ===")

require(FAST_TEST_MODE is True,
        "Smoke test requires FAST_TEST_MODE=True.")

SMOKE_SEED = 20261101
SMOKE_EQUIL_PS = 10.0
SMOKE_PRODUCTION_PS = 50.0
SMOKE_FRAME_PS = 1.0

SMOKE_BASE = OUTPUT_BASE / "smoke_tests"

SMOKE_RUN = SMOKE_BASE / datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ_smoke"
)

SMOKE_RUN.mkdir(parents=True, exist_ok=False)

print("Output :", SMOKE_RUN)
print("Seed   :", SMOKE_SEED)

# ------------------------------------------------------------
# Fresh physical objects
# ------------------------------------------------------------

smoke_system = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_system"].read_text(encoding="utf-8")
)

smoke_integrator = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_integrator"].read_text(encoding="utf-8")
)

smoke_state = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_final"].read_text(encoding="utf-8")
)

require(
    isinstance(smoke_integrator, mm.LangevinMiddleIntegrator),
    "Unexpected integrator."
)

smoke_integrator.setRandomNumberSeed(SMOKE_SEED)

barostats = [
    f for f in smoke_system.getForces()
    if isinstance(f, mm.MonteCarloBarostat)
]

require(len(barostats) == 1, "Expected one barostat.")
require(barostats[0].getFrequency() > 0,
        "Barostat is not active.")

barostats[0].setRandomNumberSeed(SMOKE_SEED + 100)

dt_ps = smoke_integrator.getStepSize().value_in_unit(
    unit.picosecond
)

require(abs(dt_ps - 0.002) < 1e-10,
        "Unexpected timestep.")

def steps(ps):
    n = round(ps / dt_ps)
    require(
        n > 0 and abs(n * dt_ps - ps) < 1e-8,
        f"Duration {ps} ps incompatible with timestep."
    )
    return n

equil_steps = steps(SMOKE_EQUIL_PS)
production_steps = steps(SMOKE_PRODUCTION_PS)
frame_interval = steps(SMOKE_FRAME_PS)

require(
    production_steps % frame_interval == 0,
    "Production length is not a multiple of frame interval."
)

# ------------------------------------------------------------
# CUDA context
# ------------------------------------------------------------

cuda = mm.Platform.getPlatformByName(PLATFORM)

sim = app.Simulation(
    pdb.topology,
    smoke_system,
    smoke_integrator,
    cuda,
    PLATFORM_PROPERTIES
)

device = cuda.getPropertyValue(sim.context, "DeviceName")

print("CUDA device :", device)

# IMPORTANT: transfer only box + positions.
# Do NOT use context.setState(smoke_state).

sim.context.setPeriodicBoxVectors(
    *smoke_state.getPeriodicBoxVectors()
)

sim.context.setPositions(
    smoke_state.getPositions()
)

sim.context.setVelocitiesToTemperature(
    smoke_integrator.getTemperature(),
    SMOKE_SEED + 200
)

# ------------------------------------------------------------
# Initial QC
# ------------------------------------------------------------

initial_state = sim.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True
)

initial_pe = initial_state.getPotentialEnergy().value_in_unit(
    unit.kilojoule_per_mole
)

initial_ke = initial_state.getKineticEnergy().value_in_unit(
    unit.kilojoule_per_mole
)

require(np.isfinite(initial_pe), "Initial PE is not finite.")
require(np.isfinite(initial_ke), "Initial KE is not finite.")

print("Initial PE :", initial_pe, "kJ/mol")
print("Initial KE :", initial_ke, "kJ/mol")

# ------------------------------------------------------------
# 10 ps NPT equilibration
# ------------------------------------------------------------

print("\nNPT equilibration:", SMOKE_EQUIL_PS, "ps")

eq_interval = steps(1.0)

sim.reporters = [
    app.StateDataReporter(
        str(SMOKE_RUN / "equilibration.csv"),
        eq_interval,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        temperature=True,
        density=True,
        speed=True,
        separator=","
    )
]

sim.step(equil_steps)
sim.reporters = []

equilibrated_state = sim.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True
)

(SMOKE_RUN / "equilibrated.xml").write_text(
    mm.XmlSerializer.serialize(equilibrated_state),
    encoding="utf-8"
)

# ------------------------------------------------------------
# Production clock reset
# ------------------------------------------------------------

sim.context.setTime(0 * unit.picosecond)
sim.currentStep = 0

# ------------------------------------------------------------
# 50 ps production
# Save SOLUTE ONLY: first 9332 atoms
# ------------------------------------------------------------

print("Production:", SMOKE_PRODUCTION_PS, "ps")

production_log = SMOKE_RUN / "production.csv"
solute_dcd = SMOKE_RUN / "solute.dcd"

sim.reporters = [
    app.DCDReporter(
        str(solute_dcd),
        frame_interval,
        atomSubset=list(range(len(atom_map))),
        enforcePeriodicBox=False
    ),

    app.StateDataReporter(
        str(production_log),
        frame_interval,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        temperature=True,
        density=True,
        speed=True,
        separator=","
    )
]

start = time.perf_counter()

sim.step(production_steps)

elapsed = time.perf_counter() - start

sim.reporters = []

# ------------------------------------------------------------
# Final QC
# ------------------------------------------------------------

final_state_smoke = sim.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True
)

final_pe = final_state_smoke.getPotentialEnergy().value_in_unit(
    unit.kilojoule_per_mole
)

final_ke = final_state_smoke.getKineticEnergy().value_in_unit(
    unit.kilojoule_per_mole
)

final_positions = final_state_smoke.getPositions(
    asNumpy=True
).value_in_unit(unit.nanometer)

require(np.isfinite(final_pe), "Final PE is not finite.")
require(np.isfinite(final_ke), "Final KE is not finite.")
require(np.isfinite(final_positions).all(),
        "Final coordinates contain NaN/Inf.")

(SMOKE_RUN / "final.xml").write_text(
    mm.XmlSerializer.serialize(final_state_smoke),
    encoding="utf-8"
)

sim.saveCheckpoint(
    str(SMOKE_RUN / "final.chk")
)

# ------------------------------------------------------------
# Provenance
# ------------------------------------------------------------

smoke_provenance = {
    "candidate": "MOL00484",
    "stage": "09o",
    "mode": "cuda_smoke_test",
    "source_09l": str(SOURCE_RUN),
    "source_sha256": SOURCE_SHA256,
    "seed": SMOKE_SEED,
    "equilibration_ps": SMOKE_EQUIL_PS,
    "production_ps": SMOKE_PRODUCTION_PS,
    "frame_ps": SMOKE_FRAME_PS,
    "platform": PLATFORM,
    "platform_properties": PLATFORM_PROPERTIES,
    "device": device,
    "initial_PE_kJ_mol": initial_pe,
    "initial_KE_kJ_mol": initial_ke,
    "final_PE_kJ_mol": final_pe,
    "final_KE_kJ_mol": final_ke,
    "elapsed_seconds": elapsed,
    "technical_qc_pass": True,
    "shared_start":
        "official 09l final 1 ns state; new stochastic velocities",
    "site_context":
        "native EGF-EGFR interface site-relevant MOL00484 configuration",
}

dump(
    SMOKE_RUN / "provenance.json",
    smoke_provenance
)

print("\nFinal PE :", final_pe, "kJ/mol")
print("Final KE :", final_ke, "kJ/mol")
print("Elapsed  :", elapsed, "s")
print("DCD      :", solute_dcd)
print("Finite coordinates:", True)

print("\n09o CUDA SMOKE TEST: TECHNICAL QC PASS")

# ------------------------------------------------------------
# Cleanup GPU context
# ------------------------------------------------------------

del sim
del smoke_integrator
del smoke_system
gc.collect()

=== 09o CUDA SMOKE TEST ===
Output : C:\Users\Roberto\PEARL\outputs\pipeline_5_09o_mol00484_bound_replicates\smoke_tests\20261028T114051_492484Z_smoke
Seed   : 20261101
CUDA device : NVIDIA GeForce RTX 4090
Initial PE : -4305893.27231567 kJ/mol
Initial KE : 790225.797779958 kJ/mol

NPT equilibration: 10.0 ps
Production: 50.0 ps

Final PE : -4306804.239286378 kJ/mol
Final KE : 783003.9047849376 kJ/mol
Elapsed  : 44.95819620000111 s
DCD      : C:\Users\Roberto\PEARL\outputs\pipeline_5_09o_mol00484_bound_replicates\smoke_tests\20261028T114051_492484Z_smoke\solute.dcd
Finite coordinates: True

09o CUDA SMOKE TEST: TECHNICAL QC PASS


9375

In [10]:
# ============================================================
# 09o — CELL 6
# A26b SITE-SPECIFIC ANALYSIS OF CUDA SMOKE TEST
# Native EGF–EGFR interface
# No MD is run in this cell.
# ============================================================

print("=== 09o SMOKE A26b — EGF INTERFACE ANALYSIS ===")

SMOKE_DCD = SMOKE_RUN / "solute.dcd"

require(
    SMOKE_DCD.is_file(),
    f"Smoke trajectory not found: {SMOKE_DCD}"
)

# ------------------------------------------------------------
# Load the small solute-only trajectory
# 9332 atoms × 50 frames
# ------------------------------------------------------------

smoke_traj = mt.load(
    str(SMOKE_DCD),
    top=str(SOURCE_FILES["solute_reference"])
)

print("\nTrajectory:")
print("  frames :", smoke_traj.n_frames)
print("  atoms  :", smoke_traj.n_atoms)

require(
    smoke_traj.n_atoms == 9332,
    "Unexpected smoke solute atom count."
)

require(
    smoke_traj.n_frames == 50,
    f"Expected 50 smoke frames, found {smoke_traj.n_frames}."
)

# ------------------------------------------------------------
# Discard first 20%
# 50 frames -> discard 10 -> analyze 40
# ------------------------------------------------------------

discard_n = int(
    np.floor(smoke_traj.n_frames * DISCARD_FRACTION)
)

analysis_traj = smoke_traj[discard_n:]

print("\nDiscard:")
print("  fraction        :", DISCARD_FRACTION)
print("  discarded frames:", discard_n)
print("  analyzed frames :", analysis_traj.n_frames)

require(
    analysis_traj.n_frames == 40,
    "Expected 40 analyzed smoke frames."
)

# ------------------------------------------------------------
# Build all interface-heavy × ligand-heavy pairs
# 344 × 25 = 8600
# ------------------------------------------------------------

pairs = np.array(
    [
        [int(i), int(j)]
        for i in interface_heavy
        for j in ligand_heavy
    ],
    dtype=int
)

require(
    pairs.shape == (8600, 2),
    f"Unexpected pair matrix: {pairs.shape}"
)

# ------------------------------------------------------------
# Periodic minimum-image distances
# ------------------------------------------------------------

distances = mt.compute_distances(
    analysis_traj,
    pairs,
    periodic=True
)

require(
    distances.shape == (40, 8600),
    f"Unexpected distance matrix: {distances.shape}"
)

require(
    np.isfinite(distances).all(),
    "Non-finite interface distances detected."
)

# Minimum ligand-interface distance in each frame
frame_min_nm = distances.min(axis=1)

contact_per_frame = (
    frame_min_nm <= INTERFACE_CUTOFF_NM
)

contact_fraction = contact_per_frame.mean()

print("\nGlobal site-specific persistence:")
print(
    "  EGF-interface contact fraction :",
    f"{contact_fraction:.6f}"
)
print(
    "  minimum distance               :",
    f"{frame_min_nm.min() * 10:.3f} Å"
)
print(
    "  mean minimum distance          :",
    f"{frame_min_nm.mean() * 10:.3f} Å"
)
print(
    "  median minimum distance        :",
    f"{np.median(frame_min_nm) * 10:.3f} Å"
)

# ------------------------------------------------------------
# Residue-specific occupancy
# ------------------------------------------------------------

rows = []

for key in sorted(interface_residue_atoms):

    residue_atoms = interface_residue_atoms[key]

    residue_pairs = np.array(
        [
            [int(i), int(j)]
            for i in residue_atoms
            for j in ligand_heavy
        ],
        dtype=int
    )

    d = mt.compute_distances(
        analysis_traj,
        residue_pairs,
        periodic=True
    )

    residue_frame_min = d.min(axis=1)

    residue_contact = (
        residue_frame_min <= INTERFACE_CUTOFF_NM
    )

    rows.append({
        "residue": key,
        "contact_frames": int(residue_contact.sum()),
        "analyzed_frames": int(analysis_traj.n_frames),
        "occupancy": float(residue_contact.mean()),
        "min_distance_A":
            float(residue_frame_min.min() * 10),
        "mean_min_distance_A":
            float(residue_frame_min.mean() * 10),
    })

residue_df = pd.DataFrame(rows)

residue_df = residue_df.sort_values(
    ["occupancy", "residue"],
    ascending=[False, True]
).reset_index(drop=True)

contacted = residue_df[
    residue_df["occupancy"] > 0
]

persistent = residue_df[
    residue_df["occupancy"] >= 0.50
]

print("\nResidue-level contacts:")
print(
    "  residues contacted at least once :",
    len(contacted)
)
print(
    "  residues occupancy >= 50%         :",
    len(persistent)
)

if len(contacted):
    print("\nContacted interface residues:")
    print(
        contacted[
            [
                "residue",
                "contact_frames",
                "occupancy",
                "min_distance_A",
                "mean_min_distance_A",
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# Save smoke A26b results
# ------------------------------------------------------------

trajectory_df = pd.DataFrame({
    "analysis_frame": np.arange(
        analysis_traj.n_frames,
        dtype=int
    ),
    "source_frame": np.arange(
        discard_n,
        smoke_traj.n_frames,
        dtype=int
    ),
    "min_interface_distance_A":
        frame_min_nm * 10.0,
    "interface_contact":
        contact_per_frame.astype(bool),
})

trajectory_df.to_csv(
    SMOKE_RUN / "A26b_EGF_interface_trajectory.csv",
    index=False
)

residue_df.to_csv(
    SMOKE_RUN / "A26b_EGF_interface_residue_occupancy.csv",
    index=False
)

a26b_summary = {
    "candidate": "MOL00484",
    "stage": "09o_smoke",
    "interface":
        "native EGF-EGFR interface",
    "interface_residues_expected":
        len(EGF_INTERFACE_RESIDUES_B),
    "interface_residues_found":
        len(found_interface),
    "interface_heavy_atoms":
        len(interface_heavy),
    "ligand_heavy_atoms":
        len(ligand_heavy),
    "pair_count":
        int(len(pairs)),
    "cutoff_nm":
        INTERFACE_CUTOFF_NM,
    "total_frames":
        int(smoke_traj.n_frames),
    "discard_fraction":
        DISCARD_FRACTION,
    "analyzed_frames":
        int(analysis_traj.n_frames),
    "contact_fraction":
        float(contact_fraction),
    "minimum_distance_A":
        float(frame_min_nm.min() * 10),
    "mean_minimum_distance_A":
        float(frame_min_nm.mean() * 10),
    "median_minimum_distance_A":
        float(np.median(frame_min_nm) * 10),
    "residues_contacted_at_least_once":
        int(len(contacted)),
    "residues_occupancy_ge_0_50":
        int(len(persistent)),
}

dump(
    SMOKE_RUN / "A26b_summary.json",
    a26b_summary
)

print("\nSaved:")
print(
    " ",
    SMOKE_RUN / "A26b_EGF_interface_trajectory.csv"
)
print(
    " ",
    SMOKE_RUN / "A26b_EGF_interface_residue_occupancy.csv"
)
print(
    " ",
    SMOKE_RUN / "A26b_summary.json"
)

print("\n09o SMOKE A26b ANALYSIS: COMPLETE")

=== 09o SMOKE A26b — EGF INTERFACE ANALYSIS ===

Trajectory:
  frames : 50
  atoms  : 9332

Discard:
  fraction        : 0.2
  discarded frames: 10
  analyzed frames : 40

Global site-specific persistence:
  EGF-interface contact fraction : 1.000000
  minimum distance               : 2.887 Å
  mean minimum distance          : 3.288 Å
  median minimum distance        : 3.324 Å

Residue-level contacts:
  residues contacted at least once : 9
  residues occupancy >= 50%         : 4

Contacted interface residues:
residue  contact_frames  occupancy  min_distance_A  mean_min_distance_A
 HIS409              40      1.000        2.996034             3.527983
  SER11              40      1.000        2.886852             3.345498
  LYS13              36      0.900        3.274418             3.754914
 LEU325              33      0.825        3.391428             4.147544
  LEU17              18      0.450        3.550638             4.584499
  ASN12               9      0.225        3.578938    

In [13]:
# ============================================================
# 09o — CELL 7
# OFFICIAL PRODUCTION — 3 stochastic bound replicas × 10 ns
#
# IMPORTANT:
# This cell starts the official long CUDA production.
# Do not rerun after successful completion unless a NEW
# independent production is explicitly intended.
# ============================================================

import time
import gc

print("=== 09o OFFICIAL BOUND PRODUCTION ===")

require(RUN_MD is True,
        "RUN_MD=False: production not authorized.")

require(FAST_TEST_MODE is False,
        "Official production requires FAST_TEST_MODE=False.")

require(len(SEEDS) == 3,
        "Official production requires exactly 3 seeds.")

require(PRODUCTION_PS == 10000.0,
        "Official production must be 10 ns per replica.")

require(EQUIL_PS == 100.0,
        "Official production requires 100 ps NPT equilibration.")

require(FRAME_PS == 10.0,
        "Official production requires 10 ps frame interval.")

# ------------------------------------------------------------
# Create official run directory
# ------------------------------------------------------------

RUN = OUTPUT_BASE / datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ_production"
)

RUN.mkdir(parents=True, exist_ok=False)

print("Official RUN :", RUN)
print("Replicas     :", len(SEEDS))
print("Seeds        :", SEEDS)
print("NPT / replica:", EQUIL_PS, "ps")
print("MD / replica :", PRODUCTION_PS, "ps")
print("Frame every  :", FRAME_PS, "ps")

# ------------------------------------------------------------
# Fixed source state:
# every replica branches from the SAME final 09l state
# ------------------------------------------------------------

initial_state = mm.XmlSerializer.deserialize(
    SOURCE_FILES["production_final"].read_text(encoding="utf-8")
)

dt_ps = 0.002

def nsteps(ps):
    n = round(ps / dt_ps)

    require(
        n > 0 and abs(n * dt_ps - ps) < 1e-8,
        f"{ps} ps incompatible with dt={dt_ps} ps."
    )

    return n

equil_steps = nsteps(EQUIL_PS)
production_steps = nsteps(PRODUCTION_PS)
frame_interval = nsteps(FRAME_PS)

require(
    production_steps % frame_interval == 0,
    "Production is not divisible by frame interval."
)

expected_frames = production_steps // frame_interval

print("Expected frames / replica :", expected_frames)

require(
    expected_frames == 1000,
    "Expected exactly 1000 frames per 10 ns replica."
)

# ------------------------------------------------------------
# Master provenance
# ------------------------------------------------------------

master_provenance = {
    "candidate": "MOL00484",
    "stage": "09o",
    "mode": "official_bound_cuda_replicates",
    "source_09l_run": str(SOURCE_RUN),
    "source_09n_run": str(FREE_RUN),
    "source_sha256": SOURCE_SHA256,
    "free_source_sha256": FREE_SHA256,
    "replica_count": len(SEEDS),
    "seeds": SEEDS,
    "equilibration_ps_per_replica": EQUIL_PS,
    "production_ps_per_replica": PRODUCTION_PS,
    "production_ns_per_replica": PRODUCTION_PS / 1000.0,
    "total_bound_production_ns":
        len(SEEDS) * PRODUCTION_PS / 1000.0,
    "frame_ps": FRAME_PS,
    "expected_frames_per_replica": expected_frames,
    "timestep_ps": dt_ps,
    "temperature_K": 300.0,
    "pressure_bar": 1.0,
    "platform": PLATFORM,
    "platform_properties": PLATFORM_PROPERTIES,
    "shared_start":
        "same official 09l final 1 ns state",
    "replica_definition":
        "stochastic branches with new velocities and random seeds",
    "microstate":
        "neutral_same_as_authoritative_09l",
    "site_context":
        "MOL00484 native EGF-EGFR interface site-relevant configuration",
    "interface_residue_count":
        len(EGF_INTERFACE_RESIDUES_B),
    "interface_cutoff_nm":
        INTERFACE_CUTOFF_NM,
    "scientific_scope":
        "geometric/dynamic persistence; not rigorous binding free energy",
}

dump(
    RUN / "provenance.json",
    master_provenance
)

# ------------------------------------------------------------
# Replica loop
# ------------------------------------------------------------

replica_records = []

for replica_number, seed in enumerate(SEEDS, start=1):

    print("\n" + "=" * 70)
    print(
        f"REPLICA {replica_number}/{len(SEEDS)} "
        f"— seed {seed}"
    )
    print("=" * 70)

    REP = RUN / f"replica_{replica_number:02d}_seed{seed}"
    REP.mkdir(parents=True, exist_ok=False)

    # --------------------------------------------------------
    # Fresh System + Integrator for every replica
    # --------------------------------------------------------

    system_rep = mm.XmlSerializer.deserialize(
        SOURCE_FILES["production_system"].read_text(
            encoding="utf-8"
        )
    )

    integrator_rep = mm.XmlSerializer.deserialize(
        SOURCE_FILES["production_integrator"].read_text(
            encoding="utf-8"
        )
    )

    require(
        isinstance(integrator_rep, mm.LangevinMiddleIntegrator),
        "Unexpected integrator."
    )

    actual_dt = integrator_rep.getStepSize().value_in_unit(
        unit.picosecond
    )

    require(
        abs(actual_dt - 0.002) < 1e-10,
        "Unexpected timestep."
    )

    # New stochastic seeds
    integrator_rep.setRandomNumberSeed(seed)

    barostats = [
        f for f in system_rep.getForces()
        if isinstance(f, mm.MonteCarloBarostat)
    ]

    require(
        len(barostats) == 1,
        "Expected exactly one MonteCarloBarostat."
    )

    require(
        barostats[0].getFrequency() > 0,
        "Barostat is not active."
    )

    barostats[0].setRandomNumberSeed(seed + 100000)

    # No positional restraints
    ext = [
        f for f in system_rep.getForces()
        if isinstance(f, mm.CustomExternalForce)
    ]

    require(
        len(ext) == 0,
        "Unexpected CustomExternalForce."
    )

    # --------------------------------------------------------
    # CUDA
    # --------------------------------------------------------

    cuda = mm.Platform.getPlatformByName(PLATFORM)

    sim_rep = app.Simulation(
        pdb.topology,
        system_rep,
        integrator_rep,
        cuda,
        PLATFORM_PROPERTIES
    )

    device = cuda.getPropertyValue(
        sim_rep.context,
        "DeviceName"
    )

    print("CUDA device :", device)

    # --------------------------------------------------------
    # SAME structural starting point for all replicas.
    #
    # IMPORTANT:
    # Do NOT call context.setState(initial_state).
    # Transfer only box + positions.
    # --------------------------------------------------------

    sim_rep.context.setPeriodicBoxVectors(
        *initial_state.getPeriodicBoxVectors()
    )

    sim_rep.context.setPositions(
        initial_state.getPositions()
    )

    # New stochastic velocities
    sim_rep.context.setVelocitiesToTemperature(
        integrator_rep.getTemperature(),
        seed + 200000
    )

    # --------------------------------------------------------
    # Initial QC
    # --------------------------------------------------------

    s0 = sim_rep.context.getState(
        getPositions=True,
        getVelocities=True,
        getEnergy=True
    )

    initial_pe = s0.getPotentialEnergy().value_in_unit(
        unit.kilojoule_per_mole
    )

    initial_ke = s0.getKineticEnergy().value_in_unit(
        unit.kilojoule_per_mole
    )

    require(
        np.isfinite(initial_pe),
        "Initial potential energy is not finite."
    )

    require(
        np.isfinite(initial_ke),
        "Initial kinetic energy is not finite."
    )

    print("Initial PE :", initial_pe, "kJ/mol")
    print("Initial KE :", initial_ke, "kJ/mol")

    # --------------------------------------------------------
    # 100 ps NPT re-equilibration
    # --------------------------------------------------------

    equil_report_interval = nsteps(10.0)

    sim_rep.reporters = [
        app.StateDataReporter(
            str(REP / "equilibration.csv"),
            equil_report_interval,
            step=True,
            time=True,
            potentialEnergy=True,
            kineticEnergy=True,
            temperature=True,
            density=True,
            speed=True,
            separator=","
        )
    ]

    print("NPT equilibration :", EQUIL_PS, "ps")

    t_eq = time.perf_counter()

    sim_rep.step(equil_steps)

    equil_elapsed = time.perf_counter() - t_eq

    sim_rep.reporters = []

    equilibrated_state = sim_rep.context.getState(
        getPositions=True,
        getVelocities=True,
        getEnergy=True
    )

    (REP / "equilibrated.xml").write_text(
        mm.XmlSerializer.serialize(equilibrated_state),
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # Reset production clock
    # --------------------------------------------------------

    sim_rep.context.setTime(0 * unit.picosecond)
    sim_rep.currentStep = 0

    # --------------------------------------------------------
    # Production reporters
    #
    # IMPORTANT:
    # DCD contains ONLY the 9332 solute atoms.
    # This avoids the huge solvated DCD problem seen previously.
    # --------------------------------------------------------

    solute_dcd = REP / "production_solute.dcd"
    production_csv = REP / "production.csv"
    checkpoint_file = REP / "production.chk"

    sim_rep.reporters = [
        app.DCDReporter(
            str(solute_dcd),
            frame_interval,
            atomSubset=list(range(len(atom_map))),
            enforcePeriodicBox=False
        ),

        app.StateDataReporter(
            str(production_csv),
            frame_interval,
            step=True,
            time=True,
            potentialEnergy=True,
            kineticEnergy=True,
            temperature=True,
            density=True,
            speed=True,
            separator=","
        ),

        app.CheckpointReporter(
            str(checkpoint_file),
            nsteps(100.0)
        ),
    ]

    print("Production :", PRODUCTION_PS / 1000.0, "ns")

    t_prod = time.perf_counter()

    sim_rep.step(production_steps)

    production_elapsed = time.perf_counter() - t_prod

    sim_rep.reporters = []

    # --------------------------------------------------------
    # Final state
    # --------------------------------------------------------

    sf = sim_rep.context.getState(
        getPositions=True,
        getVelocities=True,
        getEnergy=True
    )

    final_pe = sf.getPotentialEnergy().value_in_unit(
        unit.kilojoule_per_mole
    )

    final_ke = sf.getKineticEnergy().value_in_unit(
        unit.kilojoule_per_mole
    )

    final_positions = sf.getPositions(
        asNumpy=True
    ).value_in_unit(unit.nanometer)

    require(
        np.isfinite(final_pe),
        "Final PE is not finite."
    )

    require(
        np.isfinite(final_ke),
        "Final KE is not finite."
    )

    require(
        np.isfinite(final_positions).all(),
        "Final positions contain NaN/Inf."
    )

    (REP / "production_final.xml").write_text(
        mm.XmlSerializer.serialize(sf),
        encoding="utf-8"
    )

    sim_rep.saveCheckpoint(
        str(REP / "production_final.chk")
    )

    # --------------------------------------------------------
    # Verify trajectory exists
    # --------------------------------------------------------

    require(
        solute_dcd.is_file(),
        "Production solute DCD was not created."
    )

    require(
        production_csv.is_file(),
        "Production CSV was not created."
    )

    # --------------------------------------------------------
    # Replica provenance
    # --------------------------------------------------------

    rep_record = {
        "replica": replica_number,
        "seed": seed,
        "device": device,
        "equilibration_ps": EQUIL_PS,
        "production_ps": PRODUCTION_PS,
        "frame_ps": FRAME_PS,
        "expected_frames": expected_frames,
        "initial_PE_kJ_mol": initial_pe,
        "initial_KE_kJ_mol": initial_ke,
        "final_PE_kJ_mol": final_pe,
        "final_KE_kJ_mol": final_ke,
        "equilibration_elapsed_seconds":
            equil_elapsed,
        "production_elapsed_seconds":
            production_elapsed,
        "finite_final_coordinates": True,
        "technical_qc_pass": True,
    }

    dump(
        REP / "provenance.json",
        rep_record
    )

    replica_records.append(rep_record)

    ns_per_day = (
        (PRODUCTION_PS / 1000.0)
        / production_elapsed
        * 86400.0
    )

    print("Final PE :", final_pe, "kJ/mol")
    print("Final KE :", final_ke, "kJ/mol")
    print(
        "Production elapsed :",
        production_elapsed,
        "s"
    )
    print(
        "Approx speed       :",
        f"{ns_per_day:.2f} ns/day"
    )
    print("Replica technical QC: PASS")

    # --------------------------------------------------------
    # Release CUDA context before next replica
    # --------------------------------------------------------

    del sim_rep
    del integrator_rep
    del system_rep
    gc.collect()

# ------------------------------------------------------------
# Final production summary
# ------------------------------------------------------------

replica_df = pd.DataFrame(replica_records)

replica_df.to_csv(
    RUN / "replica_production_summary.csv",
    index=False
)

require(
    len(replica_df) == 3,
    "Not all three replicas completed."
)

require(
    replica_df["technical_qc_pass"].all(),
    "At least one replica failed technical QC."
)

final_summary = {
    "candidate": "MOL00484",
    "stage": "09o",
    "official_run": str(RUN),
    "replicas_completed": int(len(replica_df)),
    "production_ns_per_replica": 10.0,
    "total_bound_production_ns": 30.0,
    "expected_frames_per_replica": 1000,
    "all_technical_qc_pass": True,
    "next_analysis":
        "site-specific A26b persistence across all three replicas",
}

dump(
    RUN / "09o_production_summary.json",
    final_summary
)

print("\n" + "=" * 70)
print("09o OFFICIAL BOUND PRODUCTION: COMPLETE")
print("=" * 70)

print("RUN :", RUN)
print("Replicas completed :", len(replica_df))
print("Total bound MD     : 30 ns")
print("Technical QC       : PASS")
print("\nNext step: A26b analysis of all 3 × 10 ns replicas.")

=== 09o OFFICIAL BOUND PRODUCTION ===
Official RUN : C:\Users\Roberto\PEARL\outputs\pipeline_5_09o_mol00484_bound_replicates\20261028T114719_310544Z_production
Replicas     : 3
Seeds        : [20261101, 20261102, 20261103]
NPT / replica: 100.0 ps
MD / replica : 10000.0 ps
Frame every  : 10.0 ps
Expected frames / replica : 1000

REPLICA 1/3 — seed 20261101
CUDA device : NVIDIA GeForce RTX 4090
Initial PE : -4305893.272674795 kJ/mol
Initial KE : 787887.432103961 kJ/mol
NPT equilibration : 100.0 ps
Production : 10.0 ns
Final PE : -4305666.053210232 kJ/mol
Final KE : 783940.8537341778 kJ/mol
Production elapsed : 8144.976586200002 s
Approx speed       : 106.08 ns/day
Replica technical QC: PASS

REPLICA 2/3 — seed 20261102
CUDA device : NVIDIA GeForce RTX 4090
Initial PE : -4305893.272795241 kJ/mol
Initial KE : 788089.7258092354 kJ/mol
NPT equilibration : 100.0 ps
Production : 10.0 ns
Final PE : -4308648.129281156 kJ/mol
Final KE : 781982.8402539608 kJ/mol
Production elapsed : 8066.9268376 s

In [14]:
# ============================================================
# 09o — CELL 8
# OFFICIAL A26b SITE-SPECIFIC ANALYSIS
# 3 bound replicas × 10 ns
#
# Native EGF–EGFR interface
# No MD is run in this cell.
# ============================================================

print("=== 09o OFFICIAL A26b — 3 × 10 ns ===")

require(
    RUN.is_dir(),
    f"Official 09o run not found: {RUN}"
)

require(
    len(SEEDS) == 3,
    "Expected exactly 3 official replicas."
)

# ------------------------------------------------------------
# Fixed interface–ligand pair matrix
# 344 × 25 = 8600
# ------------------------------------------------------------

pairs = np.array(
    [
        [int(i), int(j)]
        for i in interface_heavy
        for j in ligand_heavy
    ],
    dtype=int
)

require(
    pairs.shape == (8600, 2),
    f"Unexpected pair matrix: {pairs.shape}"
)

all_replica_summaries = []
all_residue_tables = []
all_trajectory_tables = []

# ------------------------------------------------------------
# Analyze each replica independently
# ------------------------------------------------------------

for replica_number, seed in enumerate(SEEDS, start=1):

    print("\n" + "=" * 70)
    print(
        f"A26b REPLICA {replica_number}/3 — seed {seed}"
    )
    print("=" * 70)

    REP = RUN / f"replica_{replica_number:02d}_seed{seed}"

    DCD = REP / "production_solute.dcd"

    require(
        DCD.is_file(),
        f"Missing trajectory: {DCD}"
    )

    # --------------------------------------------------------
    # Safe load:
    # trajectory contains only 9332 solute atoms
    # --------------------------------------------------------

    traj = mt.load(
        str(DCD),
        top=str(SOURCE_FILES["solute_reference"])
    )

    print("Frames :", traj.n_frames)
    print("Atoms  :", traj.n_atoms)

    require(
        traj.n_frames == 1000,
        f"Expected 1000 frames, found {traj.n_frames}."
    )

    require(
        traj.n_atoms == 9332,
        f"Expected 9332 atoms, found {traj.n_atoms}."
    )

    # --------------------------------------------------------
    # Discard first 20%
    # 1000 -> discard 200 -> analyze 800
    # --------------------------------------------------------

    discard_n = int(
        np.floor(traj.n_frames * DISCARD_FRACTION)
    )

    analysis_traj = traj[discard_n:]

    require(
        discard_n == 200,
        f"Expected discard=200, got {discard_n}."
    )

    require(
        analysis_traj.n_frames == 800,
        "Expected 800 analyzed frames."
    )

    print("Discarded :", discard_n)
    print("Analyzed  :", analysis_traj.n_frames)

    # --------------------------------------------------------
    # Global EGF-interface distance
    # --------------------------------------------------------

    distances = mt.compute_distances(
        analysis_traj,
        pairs,
        periodic=True
    )

    require(
        distances.shape == (800, 8600),
        f"Unexpected distance matrix: {distances.shape}"
    )

    require(
        np.isfinite(distances).all(),
        "Non-finite distances detected."
    )

    frame_min_nm = distances.min(axis=1)

    contact_per_frame = (
        frame_min_nm <= INTERFACE_CUTOFF_NM
    )

    contact_fraction = float(
        contact_per_frame.mean()
    )

    # --------------------------------------------------------
    # Residue-level occupancy
    # --------------------------------------------------------

    residue_rows = []

    for key in sorted(interface_residue_atoms):

        residue_atoms = interface_residue_atoms[key]

        residue_pairs = np.array(
            [
                [int(i), int(j)]
                for i in residue_atoms
                for j in ligand_heavy
            ],
            dtype=int
        )

        d = mt.compute_distances(
            analysis_traj,
            residue_pairs,
            periodic=True
        )

        residue_frame_min = d.min(axis=1)

        residue_contact = (
            residue_frame_min <= INTERFACE_CUTOFF_NM
        )

        residue_rows.append({
            "replica": replica_number,
            "seed": seed,
            "residue": key,
            "contact_frames":
                int(residue_contact.sum()),
            "analyzed_frames":
                int(analysis_traj.n_frames),
            "occupancy":
                float(residue_contact.mean()),
            "min_distance_A":
                float(residue_frame_min.min() * 10.0),
            "mean_min_distance_A":
                float(residue_frame_min.mean() * 10.0),
        })

    residue_df_rep = pd.DataFrame(residue_rows)

    contacted = residue_df_rep[
        residue_df_rep["occupancy"] > 0
    ]

    persistent = residue_df_rep[
        residue_df_rep["occupancy"] >= 0.50
    ]

    # --------------------------------------------------------
    # Time-resolved table
    # Production frames are every 10 ps:
    # frame 200 corresponds to 2010 ps if reporter writes
    # after the first interval.
    # Use source-frame representation as primary.
    # --------------------------------------------------------

    trajectory_df_rep = pd.DataFrame({
        "replica": replica_number,
        "seed": seed,
        "analysis_frame":
            np.arange(analysis_traj.n_frames, dtype=int),
        "source_frame":
            np.arange(discard_n, traj.n_frames, dtype=int),
        "min_interface_distance_A":
            frame_min_nm * 10.0,
        "interface_contact":
            contact_per_frame.astype(bool),
    })

    # --------------------------------------------------------
    # Replica summary
    # --------------------------------------------------------

    rep_summary = {
        "replica": replica_number,
        "seed": seed,
        "total_frames": int(traj.n_frames),
        "discarded_frames": int(discard_n),
        "analyzed_frames":
            int(analysis_traj.n_frames),

        "contact_fraction":
            contact_fraction,

        "minimum_distance_A":
            float(frame_min_nm.min() * 10.0),

        "mean_minimum_distance_A":
            float(frame_min_nm.mean() * 10.0),

        "median_minimum_distance_A":
            float(np.median(frame_min_nm) * 10.0),

        "residues_contacted_at_least_once":
            int(len(contacted)),

        "residues_occupancy_ge_0_50":
            int(len(persistent)),
    }

    all_replica_summaries.append(rep_summary)
    all_residue_tables.append(residue_df_rep)
    all_trajectory_tables.append(trajectory_df_rep)

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------

    print("\nGlobal site persistence:")
    print(
        "  contact fraction       :",
        f"{contact_fraction:.6f}"
    )
    print(
        "  minimum distance       :",
        f"{rep_summary['minimum_distance_A']:.3f} Å"
    )
    print(
        "  mean minimum distance  :",
        f"{rep_summary['mean_minimum_distance_A']:.3f} Å"
    )
    print(
        "  median minimum distance:",
        f"{rep_summary['median_minimum_distance_A']:.3f} Å"
    )

    print("\nResidue contacts:")
    print(
        "  contacted >= once :",
        len(contacted)
    )
    print(
        "  occupancy >= 50%  :",
        len(persistent)
    )

    if len(contacted):

        display_df = contacted.sort_values(
            ["occupancy", "residue"],
            ascending=[False, True]
        )

        print("\nContacted residues:")
        print(
            display_df[
                [
                    "residue",
                    "contact_frames",
                    "occupancy",
                    "min_distance_A",
                    "mean_min_distance_A",
                ]
            ].to_string(index=False)
        )

    # release trajectory before next replica
    del distances
    del analysis_traj
    del traj
    gc.collect()

# ============================================================
# Combined results
# ============================================================

summary_df = pd.DataFrame(
    all_replica_summaries
)

residue_all = pd.concat(
    all_residue_tables,
    ignore_index=True
)

trajectory_all = pd.concat(
    all_trajectory_tables,
    ignore_index=True
)

print("\n" + "=" * 70)
print("09o A26b — CROSS-REPLICA SUMMARY")
print("=" * 70)

print(
    summary_df[
        [
            "replica",
            "seed",
            "contact_fraction",
            "minimum_distance_A",
            "mean_minimum_distance_A",
            "median_minimum_distance_A",
            "residues_contacted_at_least_once",
            "residues_occupancy_ge_0_50",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Aggregate frame-level persistence
# 2400 analyzed frames total
# ------------------------------------------------------------

require(
    len(trajectory_all) == 2400,
    f"Expected 2400 analyzed frames, got {len(trajectory_all)}."
)

overall_contact_fraction = float(
    trajectory_all["interface_contact"].mean()
)

overall_min_A = float(
    trajectory_all["min_interface_distance_A"].min()
)

overall_mean_A = float(
    trajectory_all["min_interface_distance_A"].mean()
)

overall_median_A = float(
    trajectory_all["min_interface_distance_A"].median()
)

print("\nAcross all 2400 analyzed frames:")
print(
    "  overall contact fraction :",
    f"{overall_contact_fraction:.6f}"
)
print(
    "  global minimum distance  :",
    f"{overall_min_A:.3f} Å"
)
print(
    "  mean minimum distance    :",
    f"{overall_mean_A:.3f} Å"
)
print(
    "  median minimum distance  :",
    f"{overall_median_A:.3f} Å"
)

# ------------------------------------------------------------
# Save official A26b results
# ------------------------------------------------------------

summary_df.to_csv(
    RUN / "09o_A26b_replica_summary.csv",
    index=False
)

residue_all.to_csv(
    RUN / "09o_A26b_residue_occupancy.csv",
    index=False
)

trajectory_all.to_csv(
    RUN / "09o_A26b_trajectory_relevance.csv",
    index=False
)

official_a26b_summary = {
    "candidate": "MOL00484",
    "stage": "09o",
    "analysis":
        "native_EGF_EGFR_interface_A26b",
    "replicas": 3,
    "production_ns_per_replica": 10.0,
    "total_bound_production_ns": 30.0,
    "discard_fraction": DISCARD_FRACTION,
    "frames_per_replica": 1000,
    "analyzed_frames_per_replica": 800,
    "total_analyzed_frames": 2400,
    "interface_residues": 43,
    "interface_heavy_atoms":
        int(len(interface_heavy)),
    "ligand_heavy_atoms":
        int(len(ligand_heavy)),
    "pair_count":
        int(len(pairs)),
    "cutoff_nm":
        INTERFACE_CUTOFF_NM,
    "overall_contact_fraction":
        overall_contact_fraction,
    "overall_minimum_distance_A":
        overall_min_A,
    "overall_mean_minimum_distance_A":
        overall_mean_A,
    "overall_median_minimum_distance_A":
        overall_median_A,
    "technical_qc_pass": True,
    "interpretation_scope":
        (
            "site-specific geometric/dynamic persistence "
            "at native EGF-EGFR interface; not rigorous "
            "DeltaG, Kd, Ki or experimental inhibition"
        ),
}

dump(
    RUN / "09o_A26b_summary.json",
    official_a26b_summary
)

print("\nSaved:")
print(" ", RUN / "09o_A26b_replica_summary.csv")
print(" ", RUN / "09o_A26b_residue_occupancy.csv")
print(" ", RUN / "09o_A26b_trajectory_relevance.csv")
print(" ", RUN / "09o_A26b_summary.json")

print("\n09o OFFICIAL A26b ANALYSIS: COMPLETE")

=== 09o OFFICIAL A26b — 3 × 10 ns ===

A26b REPLICA 1/3 — seed 20261101
Frames : 1000
Atoms  : 9332
Discarded : 200
Analyzed  : 800

Global site persistence:
  contact fraction       : 1.000000
  minimum distance       : 2.572 Å
  mean minimum distance  : 3.027 Å
  median minimum distance: 3.014 Å

Residue contacts:
  contacted >= once : 16
  occupancy >= 50%  : 5

Contacted residues:
residue  contact_frames  occupancy  min_distance_A  mean_min_distance_A
 HIS409             800    1.00000        2.753032             3.249670
  LEU17             799    0.99875        2.879192             3.585386
  ASN12             782    0.97750        2.662971             3.278138
  THR15             483    0.60375        3.101227             4.762739
  GLN16             454    0.56750        2.925436             4.645662
  SER11             304    0.38000        2.571547             4.849079
 GLN411             212    0.26500        2.699512             4.975456
 LEU325              46    0.05750  

In [2]:
# ============================================================
# 09o — CELL 9A
# Minimal recovery after kernel/server restart
# NO MD IS RUN
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import gc

PEARL_ROOT = Path(r"C:\Users\Roberto\PEARL")

# Authoritative 09n free-water run
FREE_RUN = (
    PEARL_ROOT
    / "outputs"
    / "pipeline_5_09n_mol00484_free_water"
    / "20261028T080940_432915Z_production"
)

# Authoritative 09o bound run
RUN = (
    PEARL_ROOT
    / "outputs"
    / "pipeline_5_09o_mol00484_bound_replicates"
    / "20261028T114719_310544Z_production"
)

print("=== 09o ANALYSIS RECOVERY ===")
print("09n FREE :", FREE_RUN)
print("09o BOUND:", RUN)

assert FREE_RUN.is_dir(), f"09n run not found: {FREE_RUN}"
assert RUN.is_dir(), f"09o run not found: {RUN}"

free_conf_file = FREE_RUN / "conformation_summary.csv"

assert free_conf_file.is_file(), (
    f"Missing 09n conformation summary: {free_conf_file}"
)

free_conf = pd.read_csv(free_conf_file)

print("\n=== 09n FREE-WATER CONFORMATION SUMMARY ===")
print("Shape   :", free_conf.shape)
print("Columns :", list(free_conf.columns))
print()
print(free_conf.to_string(index=False))

print("\nRECOVERY / 09n SUMMARY READ: OK")

=== 09o ANALYSIS RECOVERY ===
09n FREE : C:\Users\Roberto\PEARL\outputs\pipeline_5_09n_mol00484_free_water\20261028T080940_432915Z_production
09o BOUND: C:\Users\Roberto\PEARL\outputs\pipeline_5_09o_mol00484_bound_replicates\20261028T114719_310544Z_production

=== 09n FREE-WATER CONFORMATION SUMMARY ===
Shape   : (4, 11)
Columns : ['state', 'series', 'analyzed_frames', 'RMSD_mean_A', 'RMSD_SD_A', 'RMSD_min_A', 'RMSD_max_A', 'radius_mean_A', 'radius_SD_A', 'radius_min_A', 'radius_max_A']

state                  series  analyzed_frames  RMSD_mean_A  RMSD_SD_A  RMSD_min_A  RMSD_max_A  radius_mean_A  radius_SD_A  radius_min_A  radius_max_A
 free replica_01_seed20261029              800     1.874125   0.312686    0.508267    2.500700       4.707225     0.134492      4.247385      5.034321
 free replica_02_seed20261030              800     1.120270   0.381060    0.257716    2.286381       4.703785     0.121603      4.316650      5.034552
 free replica_03_seed20261031              800     1.3

In [4]:
# ============================================================
# 09o — CELL 9B
# MOL00484 BOUND conformational analysis
# 3 × 10 ns — directly comparable with 09n free-water
#
# NO MD IS RUN
# ============================================================

import mdtraj as mt

print("=== 09o BOUND CONFORMATIONAL ANALYSIS — 3 × 10 ns ===")

# ------------------------------------------------------------
# Authoritative source topology / atom map
# ------------------------------------------------------------

SOURCE_ROOT = PEARL_ROOT / "MOL00484_09l_WINDOWS_1ns"

TOPOLOGY_FILE = (
    SOURCE_ROOT
    / "structures"
    / "solute_reference_09f.pdb"
)

ATOM_MAP_FILE = (
    SOURCE_ROOT
    / "tables"
    / "solute_atom_map.csv"
)

assert TOPOLOGY_FILE.is_file(), TOPOLOGY_FILE
assert ATOM_MAP_FILE.is_file(), ATOM_MAP_FILE

atom_map = pd.read_csv(ATOM_MAP_FILE)

# MOL00484 heavy atoms
ligand_heavy = atom_map.loc[
    (atom_map["component"] == "MOL00484")
    & (atom_map["element"].str.upper() != "H"),
    "index"
].astype(int).to_numpy()

assert len(ligand_heavy) == 25, (
    f"Expected 25 MOL00484 heavy atoms, found {len(ligand_heavy)}"
)

print("Ligand heavy atoms :", len(ligand_heavy))

# ------------------------------------------------------------
# Official 09o replicas
# ------------------------------------------------------------

SEEDS = [20261101, 20261102, 20261103]
DISCARD_FRACTION = 0.20

bound_rows = []

for replica_number, seed in enumerate(SEEDS, start=1):

    print("\n" + "=" * 70)
    print(f"BOUND REPLICA {replica_number}/3 — seed {seed}")
    print("=" * 70)

    REP = RUN / f"replica_{replica_number:02d}_seed{seed}"
    DCD = REP / "production_solute.dcd"

    assert DCD.is_file(), DCD

    traj = mt.load(
        str(DCD),
        top=str(TOPOLOGY_FILE)
    )

    print("Frames :", traj.n_frames)
    print("Atoms  :", traj.n_atoms)

    assert traj.n_frames == 1000
    assert traj.n_atoms == 9332

    discard_n = int(
        np.floor(traj.n_frames * DISCARD_FRACTION)
    )

    traj = traj[discard_n:]

    assert traj.n_frames == 800

    print("Discarded :", discard_n)
    print("Analyzed  :", traj.n_frames)

    # --------------------------------------------------------
    # Extract MOL00484 heavy atoms only
    # --------------------------------------------------------

    lig = traj.atom_slice(ligand_heavy)

    assert lig.n_atoms == 25
    assert lig.n_frames == 800

    # --------------------------------------------------------
    # Conformational RMSD
    #
    # IMPORTANT:
    # self-fit of MOL00484 heavy atoms.
    # Each trajectory is referenced to its own first
    # analyzed frame.
    #
    # This measures internal conformational mobility,
    # NOT displacement relative to EGFR.
    # --------------------------------------------------------

    rmsd_nm = mt.rmsd(
        lig,
        lig,
        frame=0,
        atom_indices=np.arange(lig.n_atoms)
    )

    rmsd_A = rmsd_nm * 10.0

    # --------------------------------------------------------
    # Geometric radius
    #
    # Unweighted RMS distance of heavy atoms from their
    # geometric centroid.
    # This is deliberately NOT mass-weighted Rg.
    # --------------------------------------------------------

    xyz = lig.xyz  # nm

    centroid = xyz.mean(axis=1, keepdims=True)

    radius_nm = np.sqrt(
        np.mean(
            np.sum(
                (xyz - centroid) ** 2,
                axis=2
            ),
            axis=1
        )
    )

    radius_A = radius_nm * 10.0

    assert np.isfinite(rmsd_A).all()
    assert np.isfinite(radius_A).all()

    row = {
        "state": "bound",
        "series": f"replica_{replica_number:02d}_seed{seed}",
        "analyzed_frames": int(lig.n_frames),

        "RMSD_mean_A": float(np.mean(rmsd_A)),
        "RMSD_SD_A": float(np.std(rmsd_A)),
        "RMSD_min_A": float(np.min(rmsd_A)),
        "RMSD_max_A": float(np.max(rmsd_A)),

        "radius_mean_A": float(np.mean(radius_A)),
        "radius_SD_A": float(np.std(radius_A)),
        "radius_min_A": float(np.min(radius_A)),
        "radius_max_A": float(np.max(radius_A)),
    }

    bound_rows.append(row)

    print(
        f"RMSD   : {row['RMSD_mean_A']:.6f} "
        f"± {row['RMSD_SD_A']:.6f} Å"
    )

    print(
        f"         min {row['RMSD_min_A']:.6f} Å"
        f" | max {row['RMSD_max_A']:.6f} Å"
    )

    print(
        f"Radius : {row['radius_mean_A']:.6f} "
        f"± {row['radius_SD_A']:.6f} Å"
    )

    print(
        f"         min {row['radius_min_A']:.6f} Å"
        f" | max {row['radius_max_A']:.6f} Å"
    )

    del lig
    del traj
    gc.collect()

# ------------------------------------------------------------
# Final bound table
# ------------------------------------------------------------

bound_conf = pd.DataFrame(bound_rows)

print("\n" + "=" * 70)
print("09o BOUND CONFORMATION SUMMARY")
print("=" * 70)

print(bound_conf.to_string(index=False))

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

BOUND_CONF_FILE = RUN / "09o_bound_conformation_summary.csv"

bound_conf.to_csv(
    BOUND_CONF_FILE,
    index=False
)

print("\nSaved:")
print(" ", BOUND_CONF_FILE)

print("\n09o BOUND CONFORMATIONAL ANALYSIS: COMPLETE")

=== 09o BOUND CONFORMATIONAL ANALYSIS — 3 × 10 ns ===
Ligand heavy atoms : 25

BOUND REPLICA 1/3 — seed 20261101
Frames : 1000
Atoms  : 9332
Discarded : 200
Analyzed  : 800
RMSD   : 0.959999 ± 0.328702 Å
         min 0.000000 Å | max 1.971887 Å
Radius : 4.689592 ± 0.085259 Å
         min 4.384485 Å | max 4.914090 Å

BOUND REPLICA 2/3 — seed 20261102
Frames : 1000
Atoms  : 9332
Discarded : 200
Analyzed  : 800
RMSD   : 1.407900 ± 0.183512 Å
         min 0.000000 Å | max 1.846919 Å
Radius : 4.679095 ± 0.102819 Å
         min 4.377233 Å | max 4.997274 Å

BOUND REPLICA 3/3 — seed 20261103
Frames : 1000
Atoms  : 9332
Discarded : 200
Analyzed  : 800
RMSD   : 1.309635 ± 0.379904 Å
         min 0.005167 Å | max 2.397018 Å
Radius : 4.632665 ± 0.197244 Å
         min 4.055325 Å | max 5.046756 Å

09o BOUND CONFORMATION SUMMARY
state                  series  analyzed_frames  RMSD_mean_A  RMSD_SD_A  RMSD_min_A  RMSD_max_A  radius_mean_A  radius_SD_A  radius_min_A  radius_max_A
bound replica_01_seed2

In [5]:
# ============================================================
# 09o — CELL 9C
# FINAL MOL00484 FREE (09n) vs BOUND (09o) COMPARISON
#
# 09n: 3 × 10 ns free-water
# 09o: 3 × 10 ns EGFR-bound
# A26b: native EGF–EGFR interface persistence
#
# NO MD IS RUN
# ============================================================

print("=== MOL00484 FINAL FREE vs BOUND COMPARISON ===")

# ------------------------------------------------------------
# 1. Select authoritative 09n FREE replicas only
#    (exclude historical bound_09l_1ns row)
# ------------------------------------------------------------

free_only = free_conf[
    free_conf["state"] == "free"
].copy()

assert len(free_only) == 3
assert len(bound_conf) == 3

assert free_only["analyzed_frames"].sum() == 2400
assert bound_conf["analyzed_frames"].sum() == 2400

# ------------------------------------------------------------
# 2. Load authoritative 09o A26b summary
# ------------------------------------------------------------

A26B_FILE = RUN / "09o_A26b_summary.json"

assert A26B_FILE.is_file(), A26B_FILE

with open(A26B_FILE, "r", encoding="utf-8") as f:
    a26b = json.load(f)

assert a26b["total_analyzed_frames"] == 2400

# ------------------------------------------------------------
# 3. Replica-level descriptive statistics
#
# IMPORTANT:
# Each replica contributes equally because each contains
# exactly 800 analyzed frames.
# ------------------------------------------------------------

def summarize_state(df, state_name):

    return {
        "state": state_name,
        "replicas": int(len(df)),
        "analyzed_frames_total":
            int(df["analyzed_frames"].sum()),

        "RMSD_replica_mean_A":
            float(df["RMSD_mean_A"].mean()),

        "RMSD_replica_mean_SD_A":
            float(df["RMSD_mean_A"].std(ddof=1)),

        "RMSD_mean_range_min_A":
            float(df["RMSD_mean_A"].min()),

        "RMSD_mean_range_max_A":
            float(df["RMSD_mean_A"].max()),

        "radius_replica_mean_A":
            float(df["radius_mean_A"].mean()),

        "radius_replica_mean_SD_A":
            float(df["radius_mean_A"].std(ddof=1)),

        "radius_mean_range_min_A":
            float(df["radius_mean_A"].min()),

        "radius_mean_range_max_A":
            float(df["radius_mean_A"].max()),
    }


free_summary = summarize_state(
    free_only,
    "free_water_09n"
)

bound_summary = summarize_state(
    bound_conf,
    "EGFR_bound_09o"
)

comparison_df = pd.DataFrame(
    [free_summary, bound_summary]
)

# ------------------------------------------------------------
# 4. Simple descriptive differences
#
# These are NOT free energies and NOT statistical proof
# of binding.
# ------------------------------------------------------------

delta_rmsd_A = (
    bound_summary["RMSD_replica_mean_A"]
    - free_summary["RMSD_replica_mean_A"]
)

delta_radius_A = (
    bound_summary["radius_replica_mean_A"]
    - free_summary["radius_replica_mean_A"]
)

rmsd_ratio = (
    bound_summary["RMSD_replica_mean_A"]
    / free_summary["RMSD_replica_mean_A"]
)

radius_ratio = (
    bound_summary["radius_replica_mean_A"]
    / free_summary["radius_replica_mean_A"]
)

# ------------------------------------------------------------
# 5. Console output
# ------------------------------------------------------------

print("\nReplica-level source data")
print("-" * 70)

print("\n09n FREE:")
print(
    free_only[
        [
            "series",
            "analyzed_frames",
            "RMSD_mean_A",
            "RMSD_SD_A",
            "radius_mean_A",
            "radius_SD_A",
        ]
    ].to_string(index=False)
)

print("\n09o BOUND:")
print(
    bound_conf[
        [
            "series",
            "analyzed_frames",
            "RMSD_mean_A",
            "RMSD_SD_A",
            "radius_mean_A",
            "radius_SD_A",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("STATE-LEVEL DESCRIPTIVE COMPARISON")
print("=" * 70)

print(comparison_df.to_string(index=False))

print("\nDifferences BOUND - FREE:")
print(f"  Δ mean conformational RMSD : {delta_rmsd_A:+.6f} Å")
print(f"  RMSD ratio BOUND/FREE      : {rmsd_ratio:.6f}")

print(f"  Δ mean geometric radius    : {delta_radius_A:+.6f} Å")
print(f"  radius ratio BOUND/FREE    : {radius_ratio:.6f}")

# ------------------------------------------------------------
# 6. Site-specific information from A26b
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("09o NATIVE EGF–EGFR INTERFACE PERSISTENCE")
print("=" * 70)

print(
    "Analyzed bound frames       :",
    a26b["total_analyzed_frames"]
)

print(
    "Overall contact fraction    :",
    f"{a26b['overall_contact_fraction']:.6f}"
)

print(
    "Minimum interface distance  :",
    f"{a26b['overall_minimum_distance_A']:.3f} Å"
)

print(
    "Mean minimum distance       :",
    f"{a26b['overall_mean_minimum_distance_A']:.3f} Å"
)

print(
    "Median minimum distance     :",
    f"{a26b['overall_median_minimum_distance_A']:.3f} Å"
)

# ------------------------------------------------------------
# 7. Machine-readable final interpretation
# ------------------------------------------------------------

final_summary = {

    "candidate": "MOL00484",

    "comparison":
        "09n_free_water_vs_09o_EGFR_bound",

    "sampling": {
        "free_replicas": 3,
        "bound_replicas": 3,
        "production_ns_per_replica": 10.0,
        "discard_fraction": 0.20,
        "free_analyzed_frames": 2400,
        "bound_analyzed_frames": 2400,
    },

    "free_water": free_summary,

    "EGFR_bound": bound_summary,

    "bound_minus_free": {
        "mean_conformational_RMSD_A":
            float(delta_rmsd_A),

        "RMSD_ratio":
            float(rmsd_ratio),

        "mean_geometric_radius_A":
            float(delta_radius_A),

        "radius_ratio":
            float(radius_ratio),
    },

    "native_EGF_EGFR_interface": {
        "contact_fraction":
            float(a26b["overall_contact_fraction"]),

        "minimum_distance_A":
            float(a26b["overall_minimum_distance_A"]),

        "mean_minimum_distance_A":
            float(a26b["overall_mean_minimum_distance_A"]),

        "median_minimum_distance_A":
            float(a26b["overall_median_minimum_distance_A"]),
    },

    "interpretation": {
        "conformational":
            (
                "Free and bound trajectories show comparable "
                "internal conformational mobility. The bound state "
                "does not show increased conformational disruption "
                "relative to the free-water simulations."
            ),

        "site_specific":
            (
                "Across all 2400 analyzed frames from three "
                "independent stochastic bound replicas, MOL00484 "
                "remains in geometric contact with the native "
                "EGF-EGFR interface."
            ),

        "scope_limit":
            (
                "This comparison is descriptive and does not "
                "constitute a binding free-energy calculation, "
                "Kd/Ki estimate, proof of inhibition, or "
                "experimental validation."
            ),
    },
}

# ------------------------------------------------------------
# 8. Save final outputs
# ------------------------------------------------------------

comparison_csv = (
    RUN
    / "09o_final_09n_free_vs_09o_bound_comparison.csv"
)

comparison_json = (
    RUN
    / "09o_final_09n_free_vs_09o_bound_summary.json"
)

comparison_df.to_csv(
    comparison_csv,
    index=False
)

with open(
    comparison_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:")
print(" ", comparison_csv)
print(" ", comparison_json)

print("\n" + "=" * 70)
print("FINAL INTERPRETATION")
print("=" * 70)

print(
    "MOL00484 shows comparable internal conformational mobility "
    "in free-water and EGFR-bound simulations, while the bound "
    "replicas retain persistent geometric contact with the native "
    "EGF–EGFR interface across all analyzed frames."
)

print(
    "\nThis supports robust site-specific dynamic persistence, "
    "but must not be interpreted as a rigorous binding free energy, "
    "affinity estimate, inhibition measurement, or experimental validation."
)

print("\n09n / 09o FINAL COMPARISON: COMPLETE")

=== MOL00484 FINAL FREE vs BOUND COMPARISON ===

Replica-level source data
----------------------------------------------------------------------

09n FREE:
                 series  analyzed_frames  RMSD_mean_A  RMSD_SD_A  radius_mean_A  radius_SD_A
replica_01_seed20261029              800     1.874125   0.312686       4.707225     0.134492
replica_02_seed20261030              800     1.120270   0.381060       4.703785     0.121603
replica_03_seed20261031              800     1.314390   0.342885       4.707919     0.120046

09o BOUND:
                 series  analyzed_frames  RMSD_mean_A  RMSD_SD_A  radius_mean_A  radius_SD_A
replica_01_seed20261101              800     0.959999   0.328702       4.689592     0.085259
replica_02_seed20261102              800     1.407900   0.183512       4.679095     0.102819
replica_03_seed20261103              800     1.309635   0.379904       4.632665     0.197244

STATE-LEVEL DESCRIPTIVE COMPARISON
         state  replicas  analyzed_frames_total  RM

In [6]:
from pathlib import Path

BASE = Path(
    r"C:\Users\Roberto\PEARL\outputs"
    r"\pipeline_5_09o_mol00484_bound_replicates"
)

print("BASE exists:", BASE.exists())
print("BASE:", BASE)

if BASE.exists():

    print("\n" + "=" * 80)
    print("09o DIRECTORY CONTENT")
    print("=" * 80)

    for p in sorted(BASE.rglob("*")):

        if p.is_file():

            rel = p.relative_to(BASE)
            size_kb = p.stat().st_size / 1024

            print(
                f"{str(rel):90s} "
                f"{size_kb:10.1f} KB"
            )

BASE exists: True
BASE: C:\Users\Roberto\PEARL\outputs\pipeline_5_09o_mol00484_bound_replicates

09o DIRECTORY CONTENT
20261028T114719_310544Z_production\09o_A26b_replica_summary.csv                                   0.5 KB
20261028T114719_310544Z_production\09o_A26b_residue_occupancy.csv                                 8.6 KB
20261028T114719_310544Z_production\09o_A26b_summary.json                                          0.8 KB
20261028T114719_310544Z_production\09o_A26b_trajectory_relevance.csv                             81.2 KB
20261028T114719_310544Z_production\09o_bound_conformation_summary.csv                             0.6 KB
20261028T114719_310544Z_production\09o_final_09n_free_vs_09o_bound_comparison.csv                 0.5 KB
20261028T114719_310544Z_production\09o_final_09n_free_vs_09o_bound_summary.json                   2.3 KB
20261028T114719_310544Z_production\09o_production_summary.json                                    0.4 KB
20261028T114719_310544Z_production\proven

In [7]:
from pathlib import Path
import pandas as pd
import json

RUN = Path(
    r"C:\Users\Roberto\PEARL\outputs"
    r"\pipeline_5_09o_mol00484_bound_replicates"
    r"\20261028T114719_310544Z_production"
)

files = [
    "09o_A26b_replica_summary.csv",
    "09o_bound_conformation_summary.csv",
    "09o_final_09n_free_vs_09o_bound_comparison.csv",
    "09o_A26b_residue_occupancy.csv",
]

for name in files:

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    df = pd.read_csv(RUN / name)

    print("Shape:", df.shape)
    print("Columns:")
    print(list(df.columns))

    print("\nContent:")

    # residue occupancy può essere lungo:
    # mostriamo i primi 15 ordinati per occupancy se possibile
    if name == "09o_A26b_residue_occupancy.csv":

        if "occupancy" in df.columns:
            print(
                df.sort_values(
                    "occupancy",
                    ascending=False
                ).head(15).to_string(index=False)
            )
        else:
            print(df.head(15).to_string(index=False))

    else:
        print(df.to_string(index=False))


# ------------------------------------------------------------
# TRAJECTORY RELEVANCE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("09o_A26b_trajectory_relevance.csv")
print("=" * 80)

traj = pd.read_csv(
    RUN / "09o_A26b_trajectory_relevance.csv"
)

print("Shape:", traj.shape)

print("\nColumns:")
print(list(traj.columns))

print("\nFirst 10 rows:")
print(traj.head(10).to_string(index=False))

print("\nLast 10 rows:")
print(traj.tail(10).to_string(index=False))


# ------------------------------------------------------------
# FINAL SUMMARY JSON
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("09o_final_09n_free_vs_09o_bound_summary.json")
print("=" * 80)

with open(
    RUN / "09o_final_09n_free_vs_09o_bound_summary.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

print(json.dumps(data, indent=2))


09o_A26b_replica_summary.csv
Shape: (3, 11)
Columns:
['replica', 'seed', 'total_frames', 'discarded_frames', 'analyzed_frames', 'contact_fraction', 'minimum_distance_A', 'mean_minimum_distance_A', 'median_minimum_distance_A', 'residues_contacted_at_least_once', 'residues_occupancy_ge_0_50']

Content:
 replica     seed  total_frames  discarded_frames  analyzed_frames  contact_fraction  minimum_distance_A  mean_minimum_distance_A  median_minimum_distance_A  residues_contacted_at_least_once  residues_occupancy_ge_0_50
       1 20261101          1000               200              800               1.0            2.571547                 3.027088                   3.014047                                16                           5
       2 20261102          1000               200              800               1.0            2.640958                 3.119568                   3.108469                                27                           4
       3 20261103          1000         